# Decoded rating continuations

Every rating cache under `artifacts/content/artifacts/<model>/inference/<corpus>/ratings/`
holds the raw token ids the model emitted when it was asked to rate the stakes of a prompt.
This notebook reads all of them, decodes `generated_tokens` with each model's own tokenizer,
and lands one long dataframe - one row per generated continuation - carrying enough
provenance to trace any row back to the file, prompt, and generation settings it came from.
The later sections read a 0-10 score out of each decoded string, account for every row that
does not yield one, average the surviving scores to a single float per task, and rank-correlate
that against the `arc_length_parallel` coordinate in every artifact table that carries it.

**Scope.** `CORPORA` in the configuration cell limits the whole notebook to the reference
`severity` corpus. The derived inference datasets - `context` and the six `severity_*`
variants - are excluded from the caches that are read, from the per-task scores, and from
the files the correlation scans, so the two sides of that comparison always cover the same
data. Nothing below is written specifically for `severity`: widening that list, or setting
it to `None`, brings the others back with no other change.

### What is in a cache file

`scripts/stated_stakes.py` writes one JSON per template group, named
`ratings--<template_id>-<hash>.json`, with six top-level keys:

| key | shape | meaning |
| --- | --- | --- |
| `config` | dict | the generation settings: model id, dtype, seed, `max_new_tokens`, sampling flags, rating scale |
| `prompts` | list[str] | the exact text fed to the model, one per row |
| `prompt_metadata` | list[dict] | per-row provenance: `template_id`, `task`, `source_row`, `source_text`, `phrasing`, `output_format`, `instruction_position` |
| `generated_tokens` | list[list[int]] | the continuation token ids, prompt stripped off |
| `generated_text` | list[str] | what the writer decoded at generation time |
| `input_sha256` | str | fingerprint of the rows plus settings the cache was built from |

The `prompts.json` sitting beside the caches in each `ratings/` directory is the corpus
definition rather than a cache, so the glob below matches only the `ratings--` prefix.

### Why decode again when `generated_text` is already there

`generated_text` is a convenience copy the writer produced with
`batch_decode(..., skip_special_tokens=True)`. Decoding from the ids is the reproducible
path: it is the one that stays correct if a cache is ever rewritten by a different code
path, and it lets this notebook keep a second, verbatim decode that *retains* special
tokens, which is where the end-of-turn marker becomes visible. The stored copy is carried
along as `cached_text` purely so the check further down can confirm the two agree.

## Configuration

In [ ]:
import json
import sys
from collections import Counter
from functools import lru_cache
from pathlib import Path

import pandas as pd
from IPython.display import display
from transformers import AutoTokenizer

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'scripts' / 'stated_stakes.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ARTIFACT_ROOT = ROOT / 'artifacts' / 'content' / 'artifacts'

MODELS = None            # None -> every model directory under ARTIFACT_ROOT
# The reference corpus only. Widen this list, or set it to None for every corpus, to pull
# the derived inference datasets (context, flipped, pairwise, null, magnitude,
# composition, wording) back in; nothing below is specific to severity.
CORPORA = ['severity']
WRITE_TABLE = False      # True -> also persist the three tables next to the artifacts
TABLE_NAME = 'rating_continuations.parquet'
TASK_SCORE_NAME = 'rating_task_scores.csv'
CORRELATION_NAME = 'rating_arc_length_spearman.csv'

# Every tokenizer these caches name is already in the local Hugging Face cache, so the
# decode runs without network access. Flip this to False if a model has to be fetched.
OFFLINE = True

print(f'Artifacts: {ARTIFACT_ROOT.relative_to(ROOT)}')
print(f'Exists:    {ARTIFACT_ROOT.is_dir()}')

## Finding the caches

One glob covers the whole tree. The path itself carries two provenance fields that appear
nowhere inside the JSON - the model directory and the inference corpus - so they are read
off `relative_to(ARTIFACT_ROOT)` rather than reconstructed from the config.

In [ ]:
def ratings_files(artifact_root=ARTIFACT_ROOT, models=MODELS, corpora=CORPORA):
    """Every per-template rating cache, sorted, as (model, corpus, path) triples."""
    found = []
    for path in sorted(artifact_root.glob('*/inference/*/ratings/ratings--*.json')):
        model, _, corpus = path.relative_to(artifact_root).parts[:3]
        if models is not None and model not in models:
            continue
        if corpora is not None and corpus not in corpora:
            continue
        found.append((model, corpus, path))
    if not found:
        raise FileNotFoundError(f'No rating caches under {artifact_root}.')
    return found


CACHES = ratings_files()
print(f'{len(CACHES)} cache files')
print(f'models:  {sorted({model for model, _, _ in CACHES})}')
print(f'corpora: {sorted({corpus for _, corpus, _ in CACHES})}')
display(pd.crosstab([model for model, _, _ in CACHES],
                    [corpus for _, corpus, _ in CACHES]))

## Tokenizers

Each cache names its own model in `config['model_name']`, and a run mixes five of them, so
the tokenizers are loaded lazily and memoised - loading one per file would repeat the same
five loads hundreds of times. Decoding a continuation with the wrong model's tokenizer
would silently produce plausible-looking nonsense, which is why the lookup is keyed on the
id recorded in the file and never on the directory name.

In [ ]:
@lru_cache(maxsize=None)
def tokenizer_for(model_name):
    """Load and cache one tokenizer per Hugging Face model id."""
    return AutoTokenizer.from_pretrained(model_name, local_files_only=OFFLINE)


MODEL_IDS = sorted({json.loads(path.read_text(encoding='utf-8'))['config']['model_name']
                    for _, _, path in CACHES})
for model_id in MODEL_IDS:
    tokenizer = tokenizer_for(model_id)
    print(f'{model_id:<32} vocab {len(tokenizer):>7,}  eos {tokenizer.eos_token!r}')

## Decoding one cache

`batch_decode` runs twice over the same ids: once dropping special tokens, which is the
readable continuation, and once keeping them, which shows where the model actually stopped.
Both come from the same `generated_tokens`, so `n_tokens` is the true length of the
continuation rather than a property of either string.

The two columns that describe a row's origin outside this file - `prompt` and `source_text`
- differ on purpose: `prompt` is the full text the model saw, rating instruction included,
while `source_text` is the underlying scenario before the instruction was attached.

In [ ]:
PROVENANCE_FIELDS = ('template_id', 'task', 'source_row', 'phrasing',
                     'output_format', 'instruction_position')
CONFIG_FIELDS = ('model_name', 'dtype', 'seed', 'max_new_tokens', 'min_new_tokens',
                 'do_sample', 'enable_thinking', 'add_generation_prompt',
                 'rating_schema_version')


def decode_cache(model, corpus, path, artifact_root=ARTIFACT_ROOT):
    """Return one record per continuation in a single rating cache."""
    payload = json.loads(path.read_text(encoding='utf-8'))
    config = payload['config']
    tokens = payload['generated_tokens']
    metadata = payload['prompt_metadata']
    if not len(tokens) == len(metadata) == len(payload['prompts']):
        raise ValueError(f'Ragged cache: {path}')

    tokenizer = tokenizer_for(config['model_name'])
    decoded = tokenizer.batch_decode(tokens, skip_special_tokens=True)
    verbatim = tokenizer.batch_decode(tokens, skip_special_tokens=False)

    rating_scale = config.get('rating_scale') or [None, None]
    records = []
    for index, meta in enumerate(metadata):
        records.append(dict(
            model=model,
            corpus=corpus,
            row=index,
            **{field: meta.get(field) for field in PROVENANCE_FIELDS},
            generated_text=decoded[index],
            generated_text_verbatim=verbatim[index],
            cached_text=payload['generated_text'][index],
            n_tokens=len(tokens[index]),
            generated_tokens=tokens[index],
            prompt=payload['prompts'][index],
            source_text=meta.get('source_text'),
            **{field: config.get(field) for field in CONFIG_FIELDS},
            rating_scale_min=rating_scale[0],
            rating_scale_max=rating_scale[-1],
            system_prompt=config.get('system_prompt'),
            input_sha256=payload['input_sha256'],
            ratings_file=path.relative_to(artifact_root).as_posix(),
        ))
    return records


example = decode_cache(*CACHES[0])
print(f'{CACHES[0][2].name}: {len(example)} rows')
print(json.dumps({key: value for key, value in example[0].items()
                  if key not in ('prompt', 'source_text')}, indent=2, ensure_ascii=False))

## The full table

Reading and decoding all of the caches takes a few seconds; the tokenizers themselves are
the only slow part and they are already warm from the cell above. Rows come out in cache
order, then are sorted into a stable, human-readable order so that repeated runs of this
notebook produce identical tables.

In [ ]:
SORT_KEYS = ['model', 'corpus', 'template_id', 'source_row', 'phrasing', 'row']


def build_table(caches=None):
    """Decode every cache into one long frame, one row per generated continuation."""
    caches = CACHES if caches is None else caches
    records, seen = [], Counter()
    for model, corpus, path in caches:
        records.extend(decode_cache(model, corpus, path))
        seen[model] += 1
    print('caches read per model: ' + ', '.join(f'{k} {v}' for k, v in sorted(seen.items())))
    frame = pd.DataFrame.from_records(records)
    return frame.sort_values(SORT_KEYS, kind='stable').reset_index(drop=True)


ratings = build_table()
print(f'{len(ratings):,} rows x {ratings.shape[1]} columns '
      f'from {ratings["ratings_file"].nunique()} cache files')
display(ratings[['model', 'corpus', 'template_id', 'phrasing', 'source_row',
                 'n_tokens', 'generated_text']].head(12))

In [ ]:
ratings.info(memory_usage='deep')

## Checks

Four things are worth confirming before the frame is used for anything.

1. **The decode reproduces the cache.** `generated_text` derived here must equal the
   `cached_text` the writer stored; a mismatch means the ids and the text in some file have
   drifted apart, and the ids are the ones to trust.
2. **No continuation is ragged.** Generation pinned `min_new_tokens` to `max_new_tokens`,
   so every row should carry exactly that many ids.
3. **Provenance is complete.** No nulls in the columns a downstream join would key on.
4. **Rows are unique.** `(model, corpus, ratings_file, row)` addresses exactly one
   continuation, so it must have no duplicates.

In [ ]:
mismatched = ratings.loc[ratings['generated_text'] != ratings['cached_text']]
print(f'1. decode == cached_text ........ {len(mismatched)} mismatches')
if len(mismatched):
    display(mismatched[['ratings_file', 'row', 'generated_text', 'cached_text']].head())

ragged = ratings.loc[ratings['n_tokens'] != ratings['max_new_tokens']]
print(f'2. n_tokens == max_new_tokens ... {len(ragged)} ragged rows')
if len(ragged):
    display(ragged.groupby('ratings_file')['n_tokens'].agg(['min', 'max', 'size']).head())

keys = ['model', 'corpus', 'template_id', 'task', 'source_row', 'phrasing',
        'output_format', 'instruction_position', 'generated_text', 'prompt']
missing = ratings[keys].isna().sum()
print(f'3. no nulls in provenance ....... {int(missing.sum())} nulls')
if missing.sum():
    display(missing.loc[missing > 0])

duplicated = ratings.duplicated(['model', 'corpus', 'ratings_file', 'row']).sum()
print(f'4. one row per continuation ..... {int(duplicated)} duplicates')

## What the continuations look like

The rating prompt asks for a bare number, but the caches keep a fixed number of new tokens
whatever the model does with them, so most continuations are a digit followed by the
beginning of an explanation that generation cut off mid-way. The verbatim column is where an
early end-of-turn token shows up, and the coverage table is the quickest way to see which
model and corpus combinations actually have caches on disk.

In [ ]:
coverage = ratings.pivot_table(index='model', columns='corpus',
                               values='generated_text', aggfunc='size', fill_value=0)
display(coverage)

print('\nFirst line of each continuation, most common:')
first_line = ratings['generated_text'].str.split('\n').str[0].str.strip()
display(first_line.value_counts().head(10).rename('rows').to_frame())

print('\nOne sample per model:')
display(ratings.groupby('model', group_keys=False)
        .head(1)[['model', 'corpus', 'template_id', 'generated_text',
                  'generated_text_verbatim']])

## Extracting a score

The continuations are not clean integers, so a score has to be read out of the string. Three
rules define it:

1. **A run of consecutive digits is one number.** `10` is ten, not a one and a zero. The
   first such run in the string is the candidate score.
2. **Numbers outside 0-10 are rejected.** The scale is `config['rating_scale']`, and a
   number outside it means the model did not answer on the scale it was given.
3. **A candidate that ends the string with a `1` is rejected.** Generation stops at a fixed
   token budget, so a `1` sitting at the very end of the string can be the visible half of a
   truncated `10`. The value is unrecoverable, so the row is dropped rather than guessed at.
   The rule is deliberately narrow: it fires only when the *matched run itself* runs to the
   end of the string, as in `'Stakes: 1'`. A `1` that ends the string but is not part of the
   matched run - the stray token in `'<stakes>10</stakes>\n\n1'`, or the truncated `/10`
   denominator in `'0/1'` - leaves the candidate intact and the row is scored.

A string with no digits at all is **skipped**, and those rows are counted separately from
the two rejections: a skip means the model never produced a number, while a rejection means
it produced one that cannot be trusted.

### Three consequences worth knowing

**The first run wins, and a bad first run rejects the row.** It does not fall through to a
later number in the same string. This matches `parse_rating` in
[`scripts/corpora/inference/rating_phrasings.py`](../scripts/corpora/inference/rating_phrasings.py),
which range-checks its first regex match and returns `None` rather than searching on, and it
is the better reading of these continuations: in `'13\n0/10of\n'` the `13` is the model's
actual answer and the `0` is scaffolding from the truncated output shape, so falling through
would silently record a confident `0`.

**Decimals lose their fraction.** Rule 1 is about digit runs, so `'8.5'` scores as `8`, and
the tally below counts how many rows that affects. The existing `\d{1,2}` patterns in the
pipeline already behave this way, so the scores here stay comparable with the ones it
produces.

**Rule 3 is narrow on purpose, and the narrowing is what keeps the phrasings comparable.**
Tying it to the whole string instead of the matched run would reject every row ending in a
`1`, which here is mostly rows where the rating is perfectly legible: `'<stakes>10</stakes>\n\n1'`
is a clean `10` with a stray token after it, and `'0/1'` is a `0` whose `/10` **denominator**
was cut. Because it is the denominator that gets cut, those losses would fall almost entirely
on the `fraction` phrasing and quietly skew any phrasing-to-phrasing comparison. Scoped to
the matched run, the rule rejects only the handful of continuations that stop exactly where a
`0` would have come next - `'Stakes: 1'`, `'<stakes>1'`, `'N/1'` - and the cell below reports
how many rows the broader reading would have cost, so the choice stays visible.

In [ ]:
import re

# [0-9] rather than \d: the rule is about ASCII digit runs, and \d would also match
# digits from other scripts that int() accepts but the rating scale never meant.
DIGIT_RUN = re.compile(r'[0-9]+')

SCORE_MIN = int(ratings['rating_scale_min'].iloc[0])
SCORE_MAX = int(ratings['rating_scale_max'].iloc[0])
STATUSES = ('scored', 'no_number', 'out_of_range', 'trailing_one')


def extract_score(text, low=SCORE_MIN, high=SCORE_MAX):
    """Return (score, status) for one continuation under the three rules above."""
    match = DIGIT_RUN.search(text)
    if match is None:
        return None, 'no_number'          # skipped: the model never gave a number
    value = int(match.group())
    if not low <= value <= high:
        return None, 'out_of_range'       # rejected: off the stated scale
    if match.end() == len(text) and match.group().endswith('1'):
        return None, 'trailing_one'       # rejected: the run may be a truncated 10
    return value, 'scored'


CASES = ['10\n\nThe stakes are high', '8.5\n\nWait, I need to follow', 'N/10', '7',
         '{"stakes": 12}\n```', '<stakes>10</stakes>\n\n1', '0/1', 'Stakes: 1of1',
         'Stakes: 1', '<stakes>1', 'N/1', '31\nStakes: 1of1',
         'Clean up the shards carefully with a vacuum']
display(pd.DataFrame([(repr(case), *extract_score(case)) for case in CASES],
                     columns=['continuation', 'score', 'status']))

Applied to the frame, the scores land in two new columns: `score`, a nullable integer that
is missing for every row the rules did not resolve, and `score_status`, which says why. The
status is what makes the missingness auditable - a `NaN` score on its own cannot tell you
whether the model said nothing, said `47`, or was cut off mid-number.

In [ ]:
extracted = ratings['generated_text'].map(extract_score)
ratings['score'] = pd.array([value for value, _ in extracted], dtype='Int64')
ratings['score_status'] = pd.Categorical([status for _, status in extracted],
                                         categories=STATUSES)

if ratings['score_status'].isna().any():
    raise ValueError('Every row must land in one of the four statuses.')
if not ratings.loc[ratings['score_status'] == 'scored', 'score'].between(
        SCORE_MIN, SCORE_MAX).all():
    raise ValueError('A scored row fell outside the rating scale.')
if ratings.loc[ratings['score_status'] != 'scored', 'score'].notna().any():
    raise ValueError('A rejected or skipped row kept a score.')

display(ratings.loc[ratings['score_status'] != 'scored',
                    ['model', 'phrasing', 'generated_text', 'score', 'score_status']]
        .drop_duplicates('generated_text').groupby('score_status', observed=True).head(3))

## The tally

`SKIPPED_NO_NUMBER` is the count the extraction was asked to record: rows dropped because
the continuation contained no digits at all. The two rejection counts are kept beside it so
the four numbers add up to the row count.

In [ ]:
tally = ratings['score_status'].value_counts().reindex(STATUSES, fill_value=0)
SKIPPED_NO_NUMBER = int(tally['no_number'])

if int(tally.sum()) != len(ratings):
    raise ValueError('Statuses do not partition the frame.')

print(f'rows                       {len(ratings):>7,}')
print(f'scored                     {tally["scored"]:>7,}   '
      f'({tally["scored"] / len(ratings):.1%})')
print(f'skipped, no number in text {SKIPPED_NO_NUMBER:>7,}   '
      f'({SKIPPED_NO_NUMBER / len(ratings):.2%})   <- rows skipped for having no number')
print(f'rejected, out of 0-10      {tally["out_of_range"]:>7,}')
print(f'rejected, run ends in "1"  {tally["trailing_one"]:>7,}')

decimals = ((ratings['score_status'] == 'scored')
            & ratings['generated_text'].str.match(r'(?s)\s*\W*[0-9]+\.[0-9]'))
print(f'\nscored rows whose number was a decimal, counted as its integer part: '
      f'{decimals.sum():,}')

# What the broader, string-level reading of rule 3 would have cost on top of this.
broader = ((ratings['score_status'] == 'scored')
           & ratings['generated_text'].str.endswith('1'))
print(f'\nrule 3 scoped to the matched run rather than the whole string keeps '
      f'{broader.sum():,} rows\nthat a string-level test would have rejected:')
display(ratings.loc[broader, ['phrasing', 'generated_text', 'score']]
        .drop_duplicates('generated_text').head(6))

In [ ]:
print('Parse rate by model and corpus:')
display((ratings.assign(ok=ratings['score_status'] == 'scored')
         .pivot_table(index='model', columns='corpus', values='ok', aggfunc='mean')
         .style.format('{:.1%}')))

print('Status by phrasing - the output shape the prompt demanded:')
display(pd.crosstab(ratings['phrasing'], ratings['score_status'], dropna=False))

print('Score distribution:')
display(ratings.loc[ratings['score_status'] == 'scored', 'score']
        .value_counts().sort_index().rename('rows').to_frame().T)

## One score per task

Each task is rated many times over: once per rung of its severity ladder, and once per
output phrasing at every rung. Averaging those collapses a task to a single float, which is
the unit the arc lengths are keyed by.

The average is taken **within a model and corpus**, not across them. Every arc-length file
lives at `<model>/inference/<corpus>/`, so it can only be compared against the ratings that
model gave on that corpus - pooling Qwen3-32B's ratings into a correlation against
Qwen3-8B's manifold would mean nothing. Rows the extraction rejected drop out of the mean
rather than counting as zero, so `scored_rows` records how much of each task survived.

The statistic lives in one place, `TASK_STATISTIC`, and the column it produces is called
`task_score` rather than `mean_score` so that changing it here needs no edit downstream. A
mean is the right default: it uses every rating a task received, and it separates tasks
finely enough to rank them. A mode would only ever return a value some phrasing actually
emitted, collapsing the 35 tasks onto a handful of distinct scores and tying most of them
together in every ranking below.

In [ ]:
TASK_KEYS = ['model', 'corpus', 'task']
TASK_STATISTIC = 'mean'     # how the variants of one task collapse to a single score

task_scores = (ratings.groupby(TASK_KEYS, observed=True)
               .agg(task_score=('score', TASK_STATISTIC),
                    scored_rows=('score', 'count'),
                    rows=('score', 'size'))
               .reset_index())

print(f'{len(task_scores)} (model, corpus, task) groups')
print(f'tasks with no usable score at all: {task_scores["task_score"].isna().sum()}')
display(task_scores.groupby('corpus', observed=True)
        .agg(tasks=('task', 'nunique'), rows_per_task=('rows', 'mean'),
             distinct_scores=('task_score', 'nunique'),
             task_score=('task_score', 'mean')).round(2))
display(task_scores.head(8))

## Pairing each task score with its arc length

`arc_length_parallel` is the distance along the fitted stakes surface, and it is what the
stated rating is being checked against. Every file in scope is opened, and any that does not
carry the column is ignored.

Keying the two sides together takes work, because `severity_arc_lengths.csv` **has no task
column**: its rows are identified by `(template, severity_word)`, the template being the
prompt with a `{slot}` where the severity word goes. Filling that slot reproduces the
prompt string exactly, and the ratings do carry a task per prompt, so the prompt text is
the bridge between them. The check below confirms every arc row finds a task before any
correlation is computed.

Two further routes are kept for the derived corpora that `CORPORA` currently excludes, so
that widening the filter needs no other change. They are tried in this order:

| route | used by | how |
| --- | --- | --- |
| `task column` | context, composition, magnitude, wording, `surface/rows.parquet` | used directly |
| `prompt text` | any file naming a prompt but no task | the prompt is looked up in the ratings for that model and corpus |
| `template + severity_word` | **severity**, flipped, pairwise, null | the `{slot}` is filled with the severity word, and the resulting prompt is looked up |

A file's rows are then averaged per task, matching the scores, and Spearman's rho is taken
over the tasks the two sides share. Rho is used rather than Pearson because only the
**ordering** is meant to agree: a rating is a coarse 0-10 integer and an arc length is a
continuous distance, so there is no reason for the relationship to be linear.

In [ ]:
import pyarrow.parquet as pq
from scipy.stats import spearmanr

ARC_COLUMN = 'arc_length_parallel'
MIN_TASKS = 3                       # below this, a rank correlation says nothing
SLOT = re.compile(r'\{[^{}]*\}')    # the {disease} / {severity_word} hole in a template

# (model, corpus, prompt) -> task, for the files that name neither a task nor a task id.
PROMPT_TASK = (ratings[['model', 'corpus', 'source_text', 'task']].drop_duplicates()
               .set_index(['model', 'corpus', 'source_text'])['task'])
if not PROMPT_TASK.index.is_unique:
    raise ValueError('A prompt maps to more than one task within a model and corpus.')


def table_columns(path):
    """Column names without reading the body of the file."""
    if path.suffix == '.parquet':
        return list(pq.read_schema(path).names)
    return list(pd.read_csv(path, nrows=0).columns)


def task_route(columns):
    """Pick how this file's rows can be keyed by task, and the columns that needs."""
    if 'task' in columns:
        return 'task column', ['task']
    if 'prompt' in columns:
        return 'prompt text', ['prompt']
    if 'template' in columns and 'severity_word' in columns:
        return 'template + severity_word', ['template', 'severity_word']
    return None, []


def resolve_tasks(frame, route, model, corpus):
    """Return a task label per row of an arc-length file."""
    if route == 'task column':
        return frame['task']
    if route == 'prompt text':
        prompts = frame['prompt'].astype(str)
    else:
        prompts = [SLOT.sub(lambda _: str(word), template)
                   for template, word in zip(frame['template'], frame['severity_word'])]
    key = pd.MultiIndex.from_arrays([[model] * len(frame), [corpus] * len(frame),
                                     pd.Series(prompts, dtype=object).to_numpy()])
    return pd.Series(PROMPT_TASK.reindex(key).to_numpy(), index=frame.index)

In [ ]:
def file_scope(path, artifact_root=ARTIFACT_ROOT):
    """The (model, corpus) a table belongs to; corpus is None outside inference/<corpus>/."""
    relative = path.relative_to(artifact_root)
    corpus = (relative.parts[2] if len(relative.parts) > 3
              and relative.parts[1] == 'inference' else None)
    return relative.parts[0], corpus


def arc_files(artifact_root=ARTIFACT_ROOT, models=MODELS, corpora=CORPORA):
    """Every readable table under the artifact root that carries the arc-length column.

    The same MODELS/CORPORA filter that selects the rating caches applies here, so the two
    halves of the comparison always cover the same datasets. A table that belongs to no
    rated corpus - surface/rows.parquet, whose tasks come from the surface-fitting corpus -
    is in scope only when CORPORA is None, since there is nothing to correlate it against.
    """
    found = []
    for path in sorted(artifact_root.rglob('*')):
        if not path.is_file() or path.suffix not in ('.csv', '.parquet'):
            continue
        model, corpus = file_scope(path, artifact_root)
        if models is not None and model not in models:
            continue
        if corpora is not None and corpus not in corpora:
            continue
        columns = table_columns(path)
        if ARC_COLUMN in columns:
            found.append((path, columns))
    return found


def paired_tasks(path, columns, artifact_root=ARTIFACT_ROOT):
    """Join one arc-length file to the ratings, one row per task the two sides share.

    Returns (paired, info). `paired` carries task, arc length and mean score, and is None
    when the join could not be made at all; `info` records how the file was keyed and why
    it may be empty, so callers can report a reason instead of a silent gap.
    """
    relative = path.relative_to(artifact_root)
    model, corpus = file_scope(path, artifact_root)
    route, needed = task_route(columns)
    info = dict(file=relative.as_posix(), model=model, corpus=corpus, route=route,
                arc_rows=0, file_tasks=0, unresolved_rows=0, note='')
    if route is None:
        info['note'] = 'no task, prompt or template column to key on'
        return None, info

    frame = (pd.read_parquet(path, columns=[ARC_COLUMN, *needed]) if path.suffix == '.parquet'
             else pd.read_csv(path, usecols=[ARC_COLUMN, *needed]))
    info['arc_rows'] = len(frame)
    labelled = frame.assign(task=resolve_tasks(frame, route, model, corpus).to_numpy())
    info['unresolved_rows'] = int(labelled['task'].isna().sum())
    arc = (labelled.dropna(subset=['task'])
           .groupby('task', as_index=False)[ARC_COLUMN].mean())
    info['file_tasks'] = len(arc)

    if corpus is None:
        info['note'] = 'outside inference/<corpus>/; not a rated corpus'
        return None, info
    scores = task_scores.loc[(task_scores['model'] == model)
                             & (task_scores['corpus'] == corpus)]
    if scores.empty:
        info['note'] = 'no ratings cached for this model and corpus'
        return None, info

    paired = arc.merge(scores[['task', 'task_score']], on='task').dropna()
    if len(paired) < MIN_TASKS:
        info['note'] = f'only {len(paired)} shared tasks; needs {MIN_TASKS}'
    return paired, info


def correlate(path, columns, artifact_root=ARTIFACT_ROOT):
    """Spearman's rho between one file's per-task arc length and the per-task score."""
    paired, info = paired_tasks(path, columns, artifact_root)
    result = dict(info, n_tasks=0 if paired is None else len(paired),
                  spearman_r=None, p_value=None)
    if paired is None or len(paired) < MIN_TASKS:
        return result
    rho, p_value = spearmanr(paired['task_score'], paired[ARC_COLUMN])
    result['spearman_r'] = float(rho)
    result['p_value'] = float(p_value)
    return result


FILES = arc_files()
print(f'{len(FILES)} of the tables under the artifact root carry {ARC_COLUMN!r}')
correlations = pd.DataFrame([correlate(path, columns) for path, columns in FILES])
unresolved = int(correlations['unresolved_rows'].fillna(0).sum())
print(f'arc rows that could not be keyed to a task: {unresolved}')
if unresolved:
    display(correlations.loc[correlations['unresolved_rows'] > 0,
                             ['file', 'route', 'arc_rows', 'unresolved_rows']])

### The correlations

One row per file that carried the column. `n_tasks` is the number of tasks the file and the
ratings actually share, and it is small enough in places that rho should be read with it in
view rather than on its own.

In [ ]:
display(correlations[['file', 'route', 'arc_rows', 'n_tasks', 'spearman_r', 'p_value', 'note']]
        .style.format({'spearman_r': '{:.3f}', 'p_value': '{:.2g}'}, na_rep='-'))

scored_files = correlations.dropna(subset=['spearman_r'])
print(f'{len(scored_files)} of {len(correlations)} files produced a correlation; '
      f'{len(correlations) - len(scored_files)} did not:')
for note, group in correlations.loc[correlations['spearman_r'].isna()].groupby('note'):
    print(f'  {len(group):>2}  {note}')

### Seeing it

The tables above are the record; these are the read. Every figure uses one series hue and
one surface, both taken unchanged from the validated reference palette - the series step
clears the lightness band, the chroma floor and 3:1 contrast against its own surface in
both light and dark. `DARK_MODE` swaps to the dark steps of that same palette rather than
inverting the light ones, which is why it is a choice rather than a filter.

The first figure is the correlation itself: one panel per model, one marker per task. The
y axis is shared because a 0-10 rating means the same thing everywhere; the x axis is not,
because each model has its own fitted surface and its arc lengths are not on a common
scale. No trend line is drawn - the notebook never claimed the relationship was linear,
and drawing one would assert it.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DARK_MODE = False       # True -> the same palette's dark steps, chosen for the dark surface

PALETTE = {
    'light': dict(surface='#fcfcfb', ink='#0b0b0b', ink_soft='#52514e', muted='#898781',
                  grid='#e1e0d9', axis='#c3c2b7', series='#2a78d6'),
    'dark': dict(surface='#1a1a19', ink='#ffffff', ink_soft='#c3c2b7', muted='#898781',
                 grid='#2c2c2a', axis='#383835', series='#3987e5'),
}
INK = PALETTE['dark' if DARK_MODE else 'light']
FONT = 'system-ui, -apple-system, Segoe UI, sans-serif'

# Hairline, solid, one step off the surface; ticks and labels recede to the muted ink.
AXIS = dict(showgrid=True, gridcolor=INK['grid'], gridwidth=1, zeroline=False,
            showline=True, linecolor=INK['axis'], linewidth=1, ticks='outside',
            tickcolor=INK['axis'], tickfont=dict(color=INK['muted'], size=11))
MARKER = dict(size=9, line=dict(width=2, color=INK['surface']))   # 2px surface ring


def chart_layout(height, title=None, **overrides):
    """The surface, ink and spacing every figure below shares."""
    layout = dict(
        height=height, font=dict(family=FONT, size=12, color=INK['ink_soft']),
        paper_bgcolor=INK['surface'], plot_bgcolor=INK['surface'],
        margin=dict(l=80, r=40, t=90, b=60),
        hoverlabel=dict(bgcolor=INK['surface'], bordercolor=INK['axis'],
                        font=dict(family=FONT, size=12, color=INK['ink'])))
    if title is not None:
        layout['title'] = dict(text=title, x=0, xanchor='left',
                               font=dict(size=15, color=INK['ink']))
    layout.update(overrides)
    return layout


def ink_subplot_titles(figure):
    """Subplot titles are annotations, so they need the ink tokens applied by hand."""
    for annotation in figure.layout.annotations:
        annotation.font = dict(family=FONT, size=12, color=INK['ink'])
    return figure

In [ ]:
# Pair every file once here; the figures and the rank cells below all read this frame.
paired_all = pd.concat(
    [paired.assign(model=info['model'])
     for paired, info in (paired_tasks(path, columns) for path, columns in FILES)
     if paired is not None and len(paired) >= MIN_TASKS],
    ignore_index=True)
PANEL_MODELS = sorted(paired_all['model'].unique())
PANEL_COLS = 3
RHO = correlations.set_index('model')['spearman_r']

figure = make_subplots(
    rows=-(-len(PANEL_MODELS) // PANEL_COLS), cols=PANEL_COLS, shared_yaxes=True,
    horizontal_spacing=0.05, vertical_spacing=0.17,
    subplot_titles=[f'{model}   rho {RHO[model]:.2f}' for model in PANEL_MODELS])
for index, model in enumerate(PANEL_MODELS):
    part = paired_all.loc[paired_all['model'] == model]
    figure.add_trace(
        go.Scatter(x=part[ARC_COLUMN], y=part['task_score'], mode='markers',
                   showlegend=False, customdata=part['task'],
                   marker=dict(color=INK['series'], **MARKER),
                   hovertemplate=('<b>%{customdata}</b><br>stated score %{y:.2f}'
                                  '<br>arc length %{x:.1f}<extra></extra>')),
        row=index // PANEL_COLS + 1, col=index % PANEL_COLS + 1)

figure.update_xaxes(**AXIS)
figure.update_yaxes(**AXIS, range=[-0.4, 10.4], dtick=2)
figure.update_yaxes(title=dict(text='stated score',
                               font=dict(size=11, color=INK['muted'])), col=1)
figure.update_layout(**chart_layout(
    560, 'Stated stakes against distance along the stakes manifold',
    xaxis=dict(title=dict(text='arc length (per-model scale)',
                          font=dict(size=11, color=INK['muted'])))))
ink_subplot_titles(figure).show()

In [ ]:
print('Spearman rho, model x corpus:')
display(correlations.pivot_table(index='model', columns='corpus', values='spearman_r')
        .style.format('{:.3f}', na_rep='-'))

print('Shared tasks behind each cell:')
display(correlations.pivot_table(index='model', columns='corpus', values='n_tasks',
                                 aggfunc='max', fill_value=0))

## Rank agreement, allowing an off-by-one

Spearman's rho answers how strongly the two orderings agree but not how many tasks are
actually in the right place, and it punishes a neighbour swap the same way it rewards
everything else. This cell asks the blunter question: rank the tasks by mean score, rank
them by mean arc length, and count how many land within one position of each other.

The ranks are the same average-tie ranks Spearman itself uses, so the two numbers describe
one ordering rather than two. `exact` is the share of tasks whose two ranks match outright,
`within_1` the share off by at most one place.

Both need a baseline, because a tolerance band makes any number look better. Under a random
pairing of `n` tasks a given task has 3 acceptable positions out of `n` - 2 at the ends - so
chance agreement within one place is `(3n - 2) / n²`, about 8% at `n = 35`. `lift` is
`within_1` divided by that, and it is the figure worth reading: a `within_1` of 40% is only
meaningful against the 8% a shuffle would give.

A caveat that applies to the tolerance and not to rho: ranks are only as stable as the
values behind them. Task scores are means of coarse 0-10 integers, so two tasks a tenth of a
point apart swap rank on noise, while the arc lengths are continuous and effectively never
tie. Some of the off-by-one agreement is therefore luck of the draw among tasks the ratings
cannot really separate, which is why the per-task gaps are printed too.

In [ ]:
RANK_TOLERANCE = 1          # positions of slack allowed before a task counts as misplaced
RANK_METHOD = 'average'     # the tie handling Spearman uses, so the ranks match the rho


def rank_agreement(paired, tolerance=RANK_TOLERANCE):
    """Per-task rank gap between the stated score and the arc length, and its summary."""
    ranked = paired.assign(
        score_rank=paired['task_score'].rank(method=RANK_METHOD),
        arc_rank=paired[ARC_COLUMN].rank(method=RANK_METHOD))
    ranked['rank_gap'] = (ranked['score_rank'] - ranked['arc_rank']).abs()
    ranked['within_1'] = ranked['rank_gap'] <= tolerance

    count = len(ranked)
    chance = (3 * count - 2) / count ** 2          # random pairing, ends have 2 neighbours
    within = float(ranked['within_1'].mean())
    return ranked, dict(model=ranked['model'].iloc[0], n_tasks=count,
                        exact=float((ranked['rank_gap'] == 0).mean()), within_1=within,
                        mean_gap=float(ranked['rank_gap'].mean()),
                        max_gap=float(ranked['rank_gap'].max()),
                        chance=chance, lift=within / chance)


ranked_tasks, agreement = [], []
for _, group in paired_all.groupby('model', sort=True):
    detail, summary = rank_agreement(group)
    ranked_tasks.append(detail)
    agreement.append(summary)
agreement = pd.DataFrame(agreement)
task_ranks = pd.concat(ranked_tasks, ignore_index=True)

print(f'Rank agreement within {RANK_TOLERANCE} position, '
      f'against the {agreement["chance"].iloc[0]:.1%} a random pairing would give:')
display(agreement[['model', 'n_tasks', 'exact', 'within_1', 'lift', 'mean_gap', 'max_gap']]
        .style.format({'exact': '{:.1%}', 'within_1': '{:.1%}', 'lift': '{:.1f}x',
                       'mean_gap': '{:.2f}', 'max_gap': '{:.0f}'}, na_rep='-'))

The same numbers as a picture: each panel ranks the tasks both ways and plots one against
the other. The shaded band is the off-by-one tolerance, so a marker inside it is a task the
two measures place together and the band's emptiness is the result. Tasks outside it recede
to gray - the figure is about the ones that agree, and graying the rest says so without
inventing a second hue.

In [ ]:
TASK_COUNT = int(agreement['n_tasks'].max())
BAND = dict(                                     # the |score_rank - arc_rank| <= 1 ribbon
    x=[0.5, TASK_COUNT + 0.5, TASK_COUNT + 0.5, 0.5],
    y=[0.5 - RANK_TOLERANCE, TASK_COUNT + 0.5 - RANK_TOLERANCE,
       TASK_COUNT + 0.5 + RANK_TOLERANCE, 0.5 + RANK_TOLERANCE])

figure = make_subplots(
    rows=-(-len(PANEL_MODELS) // PANEL_COLS), cols=PANEL_COLS,
    shared_yaxes=True, shared_xaxes=True, horizontal_spacing=0.05, vertical_spacing=0.17,
    subplot_titles=[f'{model}   within 1: '
                    f'{agreement.set_index("model").loc[model, "within_1"]:.0%}'
                    for model in PANEL_MODELS])
for index, model in enumerate(PANEL_MODELS):
    row, col = index // PANEL_COLS + 1, index % PANEL_COLS + 1
    part = task_ranks.loc[task_ranks['model'] == model]
    figure.add_trace(go.Scatter(**BAND, fill='toself', mode='none',
                                fillcolor=INK['grid'], hoverinfo='skip',
                                showlegend=False), row=row, col=col)
    for inside, colour, name in ((True, INK['series'], f'within {RANK_TOLERANCE}'),
                                 (False, INK['muted'], 'further off')):
        side = part.loc[part['within_1'] == inside]
        figure.add_trace(
            go.Scatter(x=side['arc_rank'], y=side['score_rank'], mode='markers',
                       name=name, legendgroup=name, showlegend=index == 0,
                       customdata=side[['task', 'rank_gap']],
                       marker=dict(color=colour, **MARKER),
                       hovertemplate=('<b>%{customdata[0]}</b><br>score rank %{y:.0f}'
                                      '<br>arc rank %{x:.0f}'
                                      '<br>gap %{customdata[1]:.0f} places<extra></extra>')),
            row=row, col=col)

figure.update_xaxes(**AXIS, range=[0, TASK_COUNT + 1], dtick=10)
figure.update_yaxes(**AXIS, range=[0, TASK_COUNT + 1], dtick=10)
figure.update_yaxes(title=dict(text='rank by stated score',
                               font=dict(size=11, color=INK['muted'])), col=1)
figure.update_layout(**chart_layout(
    580, f'Rank by stated score against rank by arc length, band = off by {RANK_TOLERANCE}',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1,
                font=dict(size=11, color=INK['ink_soft'])),
    xaxis=dict(title=dict(text='rank by arc length',
                          font=dict(size=11, color=INK['muted'])))))
ink_subplot_titles(figure).show()

### Reading this against the rho above

The two cells look like they disagree - rho near 0.78, but only one task in six placed
within a position - and they do not. Spearman is a statement about the trend, not about
placement: `rho = 1 - 6·Σd² / (n(n² - 1))`, so at `n = 35` a rho of 0.78 already implies
`Σd²` of roughly 1,570, a typical gap of about five places. The mean gaps printed above are
what that rho was always describing.

What the tolerance adds is the scale of the claim. The manifold and the stated rating agree
strongly about the broad ordering - low-stakes tasks sit low, high-stakes tasks sit high -
and barely at all about where any individual task belongs among its neighbours. At `n = 35`
an off-by-one band is only three positions wide out of thirty-five, so it is a demanding
test, and the honest summary is that these two measures are ordering the same tasks the same
general way while disagreeing on the details.

Aggregating the gap per task shows where the disagreement actually sits. A task near the top
of this table is one every model ranks differently from the manifold, which is a property of
the task rather than of any one model; a task with a large spread across models is one whose
rating is unstable instead.

In [ ]:
by_task = (task_ranks.groupby('task')
           .agg(mean_gap=('rank_gap', 'mean'), worst_gap=('rank_gap', 'max'),
                models_within_1=('within_1', 'sum'), models=('within_1', 'size'),
                task_score=('task_score', 'mean'))
           .sort_values('mean_gap', ascending=False))

print(f'Tasks the manifold and the stated rating place furthest apart '
      f'(of {len(by_task)}, averaged over {int(by_task["models"].iloc[0])} models):')
display(by_task.head(10).style.format({'mean_gap': '{:.1f}', 'worst_gap': '{:.0f}',
                                       'task_score': '{:.2f}'}))

print('\nTasks both sides agree on:')
display(by_task.tail(8).style.format({'mean_gap': '{:.1f}', 'worst_gap': '{:.0f}',
                                      'task_score': '{:.2f}'}))

print(f'\nTasks within {RANK_TOLERANCE} place for every model: '
      f'{int((by_task["models_within_1"] == by_task["models"]).sum())} of {len(by_task)}')
print(f'Tasks outside it for every model:            '
      f'{int((by_task["models_within_1"] == 0).sum())} of {len(by_task)}')

In [ ]:
order = by_task.sort_values('mean_gap')          # ascending, so the worst sits at the top
chance_gap = (TASK_COUNT ** 2 - 1) / (3 * TASK_COUNT)   # E|i - j| for a random pairing

figure = go.Figure(go.Bar(
    x=order['mean_gap'], y=order.index, orientation='h', width=0.68,
    marker=dict(color=INK['series'], cornerradius=4),
    customdata=order[['worst_gap', 'models_within_1', 'models']].to_numpy(),
    hovertemplate=('<b>%{y}</b><br>mean gap %{x:.1f} places'
                   '<br>worst %{customdata[0]:.0f}'
                   '<br>within 1 for %{customdata[1]:.0f} of %{customdata[2]:.0f} models'
                   '<extra></extra>')))
figure.add_vline(x=chance_gap, line=dict(color=INK['axis'], width=1),
                 annotation_text=f'random pairing  {chance_gap:.1f}',
                 annotation_position='top right',
                 annotation_font=dict(size=11, color=INK['muted']))
figure.update_xaxes(**AXIS, title=dict(text='mean rank gap, places',
                                       font=dict(size=11, color=INK['muted'])))
figure.update_yaxes(**dict(AXIS, showgrid=False,
                           tickfont=dict(size=10, color=INK['muted'])))
figure.update_layout(**chart_layout(
    max(420, 20 * len(order) + 140),
    'How far apart the two measures place each task, averaged over models',
    margin=dict(l=170, r=60, t=90, b=60), bargap=0.32))
figure.show()

## Saving

Off by default, since everything here rebuilds from the caches in seconds. When enabled,
three tables are written beside the per-model artifact directories: the row-level frame,
the per-task means, and the per-file correlations. The row-level frame goes to Parquet
because `generated_tokens` is a list column, which Parquet stores natively whereas a CSV
would flatten it to a string that no longer round-trips; the two summaries are small and
go to CSV so they can be read without pandas.

In [ ]:
if WRITE_TABLE:
    for frame, name in ((ratings, TABLE_NAME),
                        (task_scores, TASK_SCORE_NAME),
                        (correlations, CORRELATION_NAME)):
        path = ARTIFACT_ROOT / name
        if path.suffix == '.parquet':
            frame.to_parquet(path, index=False)
        else:
            frame.to_csv(path, index=False)
        print(f'Wrote {path.relative_to(ROOT)} ({len(frame):,} rows)')
else:
    print('WRITE_TABLE is False; nothing written.')